In [1]:
import os
import datetime
import random
import pickle

import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
data=pd.read_csv('Churn_Modelling.csv')
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

class_weights_array = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Class weights:", class_weight_dict)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()
opt = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss="binary_crossentropy",
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
    ]
)
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    class_weight=class_weight_dict,
    callbacks=[tensorboard_callback, early_stopping_callback],
    verbose=1
)
model.save('model.keras')

results = model.evaluate(X_test, y_test, verbose=0)
for metric_name, metric_value in zip(model.metrics_names, results):
    print(f"{metric_name}: {metric_value:.4f}")

Class weights: {0: 0.6279434850863422, 1: 2.4539877300613497}


2026-09-18 12:52:31.921914: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-09-18 12:52:31.921975: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-18 12:52:31.921998: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-18 12:52:31.922301: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-18 12:52:31.922687: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


Epoch 1/100


2026-09-18 12:52:33.351779: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-09-18 12:52:33.420282: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


250/250 [==============================] - 6s 19ms/step - loss: 0.6014 - accuracy: 0.6846 - precision: 0.3574 - recall: 0.6865 - auc: 0.7434 - val_loss: 0.5813 - val_accuracy: 0.7040 - val_precision: 0.3772 - val_recall: 0.6978 - val_auc: 0.7715
Epoch 2/100
250/250 [==============================] - 4s 17ms/step - loss: 0.5869 - accuracy: 0.7071 - precision: 0.3779 - recall: 0.6767 - auc: 0.7593 - val_loss: 0.5921 - val_accuracy: 0.7095 - val_precision: 0.3861 - val_recall: 0.7248 - val_auc: 0.7782
Epoch 3/100
250/250 [==============================] - 4s 16ms/step - loss: 0.5898 - accuracy: 0.7081 - precision: 0.3805 - recall: 0.6883 - auc: 0.7569 - val_loss: 0.5666 - val_accuracy: 0.7080 - val_precision: 0.3812 - val_recall: 0.6978 - val_auc: 0.7738
Epoch 4/100
250/250 [==============================] - 4s 18ms/step - loss: 0.6032 - accuracy: 0.7084 - precision: 0.3772 - recall: 0.6626 - auc: 0.7466 - val_loss: 0.5672 - val_accuracy: 0.7215 - val_precision: 0.3846 - val_recall: 0.614